# 04 - Invoke Azure Function and View Logs

This notebook invokes the Speech Function with function-key authentication, verifies that the function can call the Speech endpoint with managed identity, and then queries recent logs.

## Step 1 - Load environment and helpers

In [16]:
import json
import os
import shutil
import subprocess
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env", override=True)

def resolve_az_cli():
    az_in_path = shutil.which("az")
    if az_in_path:
        return az_in_path
    windows_fallbacks = [
        Path(r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"),
        Path(r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"),
    ]
    for candidate in windows_fallbacks:
        if candidate.exists():
            return str(candidate)
    return None

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found from this kernel. Install it or add it to PATH.")

resource_group = os.getenv("AZURE_RESOURCE_GROUP")
if not resource_group:
    raise RuntimeError("Missing AZURE_RESOURCE_GROUP in ../env/.env")

print(f"Resource group: {resource_group}")

Resource group: rg-speech01-foundry


## Step 2 - Load Function values from env

This notebook expects function metadata and the function key to already exist in ../env/.env from Notebook 1.

In [17]:
function_app_name = os.getenv("AZURE_FUNCTION_APP_NAME", "").strip()
function_url = os.getenv("AZURE_FUNCTION_URL", "").strip()
app_insights_name = os.getenv("AZURE_APP_INSIGHTS_NAME", "").strip()
function_key = os.getenv("AZURE_FUNCTION_KEY", "").strip()

if not function_app_name or not function_url or not app_insights_name or not function_key:
    raise RuntimeError(
        "Missing function metadata or function key in ../env/.env. Rerun Notebook 1 to provision infra and Notebook 2 to refresh the key."
    )

print(f"Function app: {function_app_name}")
print(f"Function URL: {function_url}")
print(f"App Insights: {app_insights_name}")
print("Function key: loaded")

Function app: func-speech01-6m4wo
Function URL: https://func-speech01-6m4wo.azurewebsites.net/api/speech-roundtrip
App Insights: appi-speech01-6m4wo
Function key: loaded


## Step 3 - Invoke function with function key

A successful response should include `auth_mode: managed_identity`, the Speech endpoint, a request id, and the synthesized audio byte count.

In [19]:
payload = {
    "text": "Managed identity lets this function call the Azure Speech endpoint without storing secrets."
}


def invoke_with_key(key_to_use: str):
    return requests.post(
        function_url,
        params={"code": key_to_use},
        headers={
            "Content-Type": "application/json",
            "x-functions-key": key_to_use,
        },
        json=payload,
        timeout=60,
    )


try:
    resp = invoke_with_key(function_key)
except requests.RequestException as ex:
    raise RuntimeError(f"Function invocation failed before receiving a response: {ex}") from ex

if resp.status_code == 401:
    print("Received HTTP 401. Refreshing function key from Azure and retrying once...")
    refresh_cmd = [
        az_cmd,
        "functionapp",
        "keys",
        "list",
        "--resource-group",
        resource_group,
        "--name",
        function_app_name,
        "--query",
        "functionKeys.default",
        "--output",
        "tsv",
    ]
    refresh_result = subprocess.run(refresh_cmd, capture_output=True, text=True)
    refreshed_key = refresh_result.stdout.strip()

    if refresh_result.returncode != 0 or not refreshed_key:
        raise RuntimeError(
            "Function returned HTTP 401 and key refresh failed.\n"
            f"stderr: {refresh_result.stderr.strip()}\n"
            f"stdout: {refresh_result.stdout.strip()}"
        )

    env_path = Path("../env/.env")
    env_text = env_path.read_text(encoding="utf-8")
    env_lines = []
    found_key = False
    for line in env_text.splitlines():
        if line.startswith("AZURE_FUNCTION_KEY="):
            env_lines.append(f"AZURE_FUNCTION_KEY={refreshed_key}")
            found_key = True
        else:
            env_lines.append(line)
    if not found_key:
        env_lines.append(f"AZURE_FUNCTION_KEY={refreshed_key}")
    env_path.write_text("\n".join(env_lines) + "\n", encoding="utf-8")

    function_key = refreshed_key
    try:
        resp = invoke_with_key(function_key)
    except requests.RequestException as ex:
        raise RuntimeError(f"Retry after key refresh failed before receiving a response: {ex}") from ex

if resp.status_code < 200 or resp.status_code >= 300:
    raise RuntimeError(
        f"Function invocation failed with HTTP {resp.status_code}.\n"
        f"Response body: {resp.text}"
    )

print(f"HTTP status: {resp.status_code}")
try:
    print(json.dumps(resp.json(), indent=2))
except Exception:
    print(resp.text)

RuntimeError: Function invocation failed with HTTP 500.
Response body: {"error": "Speech endpoint returned HTTP 404. Response body: {\"error\":{\"code\":\"404\",\"message\": \"Resource not found\"}}"}

## Step 4 - Query recent function traces

In [6]:
if not app_insights_name:
    raise RuntimeError("App Insights component not found in this resource group.")

kql = "traces | where timestamp > ago(30m) | order by timestamp desc | take 25"
query_cmd = [
    az_cmd,
    "monitor",
    "app-insights",
    "query",
    "--app",
    app_insights_name,
    "--analytics-query",
    kql,
    "--output",
    "table",
]

subprocess.run(query_cmd, check=True)

CalledProcessError: Command '['C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.CMD', 'monitor', 'app-insights', 'query', '--app', 'appi-speech01-6m4wo', '--analytics-query', 'traces | where timestamp > ago(30m) | order by timestamp desc | take 25', '--output', 'table']' returned non-zero exit status 1.